# Stable Diffusion XL 1.0 Base · Retrato animado de Liam

Prueba de imagen a imagen para PlanIA P2. SDXL recibe la foto de Liam desde un dataset privado de Kaggle y genera un retrato con estilo de animación 3D.

En Kaggle abre **Edit**, selecciona **GPU T4 ×2** en **Accelerator**, activa **Internet** y pulsa **Run All**. El dataset de Liam ya está adjunto. Puedes cambiar la imagen o el prompt en el paso 2.

[Dataset de Liam](https://www.kaggle.com/datasets/jeffreyamc/plania-p2-liam-input) · [Ficha de SDXL 1.0 Base](https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0)

## 1. Instalar dependencias

Usamos el PyTorch con CUDA que ya incluye Kaggle y fijamos las versiones de las librerías de inferencia.

In [ ]:
%pip install -q diffusers==0.35.2 transformers==4.57.1 accelerate==1.11.0

## 2. Elegir la imagen y el prompt

Con **image_path = None**, el notebook busca **liam.jpeg** en el dataset adjunto. Para usar otra foto, adjúntala con **Add Input** y copia su ruta bajo **/kaggle/input/** en **image_path**.

El prompt describe el estilo y los rasgos que se quieren conservar. **strength = 0.50** limita cuánto se transforma la entrada. Puedes bajar ese valor para conservar más rasgos o subirlo para acentuar la estilización.

In [ ]:
image_path = None  # Usa liam.jpeg del dataset adjunto.
prompt = (
    "A realistic 3D animated-film portrait of the same cat in the reference photo. "
    "Preserve its facial proportions, pale blue eyes, pink nose, cream-white fur and "
    "light orange markings around the eyes and ears. Keep the same pose and expression, "
    "with softly rendered fur and natural lighting."
)
strength = 0.50
num_inference_steps = 25
guidance_scale = 7.5
seed = 42

## 3. Cargar la foto y comprobar la GPU

Kaggle monta el dataset bajo **/kaggle/input/**. Respetamos la orientación de la foto y la ajustamos a **768 × 1024**, que mantiene las proporciones de Liam. Si usas otra imagen, añadimos relleno cuando sea necesario. El notebook usa la primera GPU detectada.

In [ ]:
import json
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import torch
from diffusers import StableDiffusionXLImg2ImgPipeline
from IPython.display import display
from PIL import Image, ImageOps

if not torch.cuda.is_available():
    raise RuntimeError("Activa una GPU en Settings → Accelerator y ejecuta de nuevo.")

device = torch.device("cuda:0")
gpu = torch.cuda.get_device_properties(device)
gib = 1024**3
print(f"GPU: {gpu.name} | VRAM visible para PyTorch: {gpu.total_memory / gib:.2f} GiB")

if image_path is None:
    image_path = next(Path("/kaggle/input").glob("**/plania-p2-liam-input/liam.jpeg"), None)
    if image_path is None:
        raise FileNotFoundError("Adjunta el dataset plania-p2-liam-input o indica image_path.")

with Image.open(image_path) as source:
    input_image = ImageOps.exif_transpose(source).convert("RGB")

input_image = ImageOps.pad(
    input_image, (768, 1024), Image.Resampling.LANCZOS, color="#f8f2e7"
)

output_dir = Path("/kaggle/working")
input_image.save(output_dir / "entrada.png")

## 4. Cargar SDXL 1.0 Base

Descargamos una revisión fija del modelo y cargamos sus pesos en la primera GPU en FP16. El modelo base genera la imagen por sí solo. La primera descarga puede tardar varios minutos.

In [ ]:
model_id = "stabilityai/stable-diffusion-xl-base-1.0"
model_revision = "462165984030d82259a11f4367a4eed129e94a7b"

torch.cuda.reset_peak_memory_stats(device)
load_start = perf_counter()
pipeline = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    model_id,
    revision=model_revision,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)
pipeline.to(device)
pipeline.set_progress_bar_config(disable=True)
torch.cuda.synchronize(device)
load_seconds = perf_counter() - load_start

## 5. Generar y guardar una variación

La imagen y el prompt guían la generación. Guardamos la entrada preparada, la variación y un JSON con los parámetros, el tiempo y el pico de memoria de PyTorch durante la carga y la generación. El tiempo de generación se mide por separado.

En una sesión interactiva encontrarás los archivos en **/kaggle/working**. Para conservarlos en la pestaña **Output**, usa **Save Version → Save & Run All**.

In [ ]:
generator = torch.Generator(device=device).manual_seed(seed)

generation_start = perf_counter()
result = pipeline(
    prompt=prompt,
    image=input_image,
    strength=strength,
    num_inference_steps=num_inference_steps,
    guidance_scale=guidance_scale,
    generator=generator,
)
torch.cuda.synchronize(device)
generation_seconds = perf_counter() - generation_start

generated_image = result.images[0]
generated_image.save(output_dir / "generada.png")

summary = {
    "status": "complete",
    "stage": "done",
    "model": model_id,
    "revision": model_revision,
    "dtype": "float16",
    "gpu": gpu.name,
    "gpu_count": torch.cuda.device_count(),
    "vram_total_gib": round(gpu.total_memory / gib, 2),
    "vram_peak_allocated_gib": round(torch.cuda.max_memory_allocated(device) / gib, 2),
    "vram_peak_reserved_gib": round(torch.cuda.max_memory_reserved(device) / gib, 2),
    "load_seconds": round(load_seconds, 2),
    "generation_seconds": round(generation_seconds, 2),
    "versions": {name: version(name) for name in ("torch", "diffusers", "transformers", "accelerate")},
    "input_source": str(image_path),
    "size_px": list(generated_image.size),
    "prompt": prompt,
    "strength": strength,
    "num_inference_steps": num_inference_steps,
    "guidance_scale": guidance_scale,
    "seed": seed,
    "artifacts": ["entrada.png", "generada.png", "run_summary.json"],
}
(output_dir / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Generación: {generation_seconds:.2f} s | Pico asignado: {summary['vram_peak_allocated_gib']:.2f} GiB")
print("Entrada a la izquierda y variación generada a la derecha.")
preview_size = (288, 384)
preview = Image.new("RGB", (2 * preview_size[0] + 8, preview_size[1]), "white")
preview.paste(input_image.resize(preview_size), (0, 0))
preview.paste(generated_image.resize(preview_size), (preview_size[0] + 8, 0))
display(preview)